# Forest Fire AI — Tabular Model Benchmarking
## Notebook 07: Benchmark ML Models for Classification & Regression


In [ ]:
import os, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import cross_val_score, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, mean_absolute_error,
                              mean_squared_error, r2_score)
import xgboost as xgb

ROOT     = Path(r"e:/Sharvayu data/Malware/Symbiosis Nagpur SIT/7th SEM/Forest Fire task")
IMPL     = ROOT / "Implementation"
META_DIR = IMPL / "artifacts" / "metadata"
PLOTS    = IMPL / "artifacts" / "plots"
DATA_DIR = IMPL / "data" / "processed"

with open(META_DIR / "tabular_metadata.json") as f:
    tmeta = json.load(f)

df = pd.read_csv(DATA_DIR / "forestfires_processed.csv")
print(f"Data: {df.shape}")
print(f"Features: {tmeta['features']}")


In [ ]:
# Preprocessing
num_feats = tmeta['numerical_features']
cat_feats  = tmeta['categorical_features']

le_month = LabelEncoder()
le_day   = LabelEncoder()
df['month_enc'] = le_month.fit_transform(df['month'])
df['day_enc']   = le_day.fit_transform(df['day'])

feature_cols = num_feats + ['month_enc', 'day_enc']
X = df[feature_cols].values
y_cls = df['fire_occurred'].values
y_reg = df['area'].values
y_reg_log = np.log1p(y_reg)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Feature matrix: {X.shape}")
print(f"Classification target distribution: {np.bincount(y_cls)}")
print(f"Regression target range: {y_reg.min():.2f} to {y_reg.max():.2f}")


In [ ]:
# ── Classification Benchmark ──────────────────────────────────────────
cls_models = {
    'XGBoost': xgb.XGBClassifier(n_estimators=100, random_state=42,
                                   eval_metric='logloss', verbosity=0),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'HistGradientBoosting': HistGradientBoostingClassifier(max_iter=100, random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000)
}

cls_results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, clf in cls_models.items():
    t0 = time.time()
    # Use scaled for logistic, raw+encoded for tree-based
    X_use = X_scaled if name == 'Logistic Regression' else X
    scores_acc = cross_val_score(clf, X_use, y_cls, cv=cv, scoring='accuracy')
    scores_f1  = cross_val_score(clf, X_use, y_cls, cv=cv, scoring='f1')
    scores_auc = cross_val_score(clf, X_use, y_cls, cv=cv, scoring='roc_auc')
    elapsed = time.time() - t0
    res = {
        'model': name,
        'cv_accuracy': round(scores_acc.mean(), 4),
        'cv_f1': round(scores_f1.mean(), 4),
        'cv_roc_auc': round(scores_auc.mean(), 4),
        'time_s': round(elapsed, 1)
    }
    cls_results.append(res)
    print(f"  {name}: acc={res['cv_accuracy']:.4f} | f1={res['cv_f1']:.4f} | auc={res['cv_roc_auc']:.4f} | {elapsed:.1f}s")

df_cls = pd.DataFrame(cls_results).sort_values('cv_f1', ascending=False)
print("\nClassification Benchmark (5-fold CV):")
print(df_cls.to_string(index=False))
best_cls = df_cls.iloc[0]['model']
print(f"\nBest classifier: {best_cls}")


In [ ]:
# ── Regression Benchmark ──────────────────────────────────────────────
reg_models = {
    'XGBoost': xgb.XGBRegressor(n_estimators=100, random_state=42, verbosity=0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'HistGradientBoosting': HistGradientBoostingRegressor(max_iter=100, random_state=42),
}

reg_results = []
cv_r = KFold(n_splits=5, shuffle=True, random_state=42)

for name, reg in reg_models.items():
    t0 = time.time()
    scores_mae  = -cross_val_score(reg, X, y_reg_log, cv=cv_r, scoring='neg_mean_absolute_error')
    scores_r2   =  cross_val_score(reg, X, y_reg_log, cv=cv_r, scoring='r2')
    elapsed = time.time() - t0
    res = {'model': name, 'cv_mae': round(scores_mae.mean(), 4),
           'cv_r2': round(scores_r2.mean(), 4), 'time_s': round(elapsed, 1)}
    reg_results.append(res)
    print(f"  {name}: MAE(log)={res['cv_mae']:.4f} | R²={res['cv_r2']:.4f} | {elapsed:.1f}s")

df_reg = pd.DataFrame(reg_results).sort_values('cv_r2', ascending=False)
print("\nRegression Benchmark (5-fold CV, log-transformed target):")
print(df_reg.to_string(index=False))
best_reg = df_reg.iloc[0]['model']
print(f"\nBest regressor: {best_reg}")


In [ ]:
# Save benchmark results and selections
bench_tab = {
    'classification': cls_results,
    'regression': reg_results,
    'best_classifier': best_cls,
    'best_regressor': best_reg,
    'feature_cols': feature_cols
}
with open(META_DIR / "tabular_benchmark.json", "w") as f:
    json.dump(bench_tab, f, indent=2)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
names_cls = [r['model'] for r in cls_results]
f1s       = [r['cv_f1'] for r in cls_results]
aucs      = [r['cv_roc_auc'] for r in cls_results]

x = np.arange(len(names_cls))
w = 0.35
bars1 = axes[0].bar(x - w/2, f1s,  w, label='F1',     color='#1E90FF', edgecolor='black')
bars2 = axes[0].bar(x + w/2, aucs, w, label='ROC-AUC', color='#32CD32', edgecolor='black')
axes[0].set_xticks(x); axes[0].set_xticklabels(names_cls, rotation=20, ha='right', fontsize=9)
axes[0].set_ylim(0, 1.1); axes[0].set_title('Classification Benchmark', fontweight='bold')
axes[0].legend(); axes[0].set_ylabel('Score')

names_reg = [r['model'] for r in reg_results]
r2s = [max(0, r['cv_r2']) for r in reg_results]
bars3 = axes[1].bar(names_reg, r2s, color='#FF8C00', edgecolor='black', alpha=0.85)
axes[1].set_title('Regression Benchmark (R²)', fontweight='bold')
axes[1].set_ylabel('R²'); axes[1].set_ylim(0, 1.0)
axes[1].set_xticklabels(names_reg, rotation=20, ha='right', fontsize=9)
for bar, val in zip(bars3, r2s):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                 f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(PLOTS / "tabular_benchmark.png", dpi=100, bbox_inches='tight')
plt.close()
print("Tabular benchmark complete. Best classifier:", best_cls, "| Best regressor:", best_reg)
